# LAB 2: PHÂN LOẠI SỐNG SÓT TITANIC (DECISION TREE & RANDOM FOREST)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## Đọc dữ liệu

In [2]:
titanic_df = pd.read_csv('../data/titanic.csv')

## Tiền xử lý dữ liệu

In [3]:
titanic_df['Age'].fillna(titanic_df['Age'].median(), inplace=True)
if 'Embarked' in titanic_df.columns:
    titanic_df['Embarked'].fillna(titanic_df['Embarked'].mode()[0], inplace=True)
    titanic_df = pd.get_dummies(titanic_df, columns=['Sex', 'Embarked'], drop_first=True)
else:
    titanic_df = pd.get_dummies(titanic_df, columns=['Sex'], drop_first=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_12280\1963984266.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  titanic_df['Age'].fillna(titanic_df['Age'].median(), inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_12280\1963984266.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through 

## Lựa chọn đặc trưng

In [4]:
features = [col for col in ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S'] if col in titanic_df.columns]
X = titanic_df[features]
y = titanic_df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Decision Tree

In [5]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print("=== DECISION TREE ACCURACY ===")
print("Accuracy:", accuracy_score(y_test, dt.predict(X_test)))

=== DECISION TREE ACCURACY ===
Accuracy: 0.7988826815642458


## Random Forest + GridSearchCV

In [6]:
rf_params = {'n_estimators': [50, 100, 150], 'max_depth': [3, 5, 7]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=rf_params, cv=5)
rf_grid.fit(X_train, y_train)

print("\n=== RANDOM FOREST BEST MODEL ===")
print("Tham số tối ưu:", rf_grid.best_params_)
print("Classification Report:\n", classification_report(y_test, rf_grid.best_estimator_.predict(X_test)))


=== RANDOM FOREST BEST MODEL ===
Tham số tối ưu: {'max_depth': 5, 'n_estimators': 100}
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.91      0.86       105
           1       0.85      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.83      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



### GIẢI THÍCH Ý NGHĨA KẾT QUẢ CHẠY:
1. **Decision Tree:** Giới hạn chiều sâu `max_depth = 4` giúp cây quyết định rút ra các quy tắc phân loại đơn giản, tránh học vẹt (overfitting).
2. **Random Forest:** Cho kết quả chính xác cao hơn nhờ kết hợp dự đoán của nhiều cây độc lập. Việc dùng `GridSearchCV` giúp tự động tìm số cây (`n_estimators`) cho chỉ số F1-Score tối ưu nhất.